In [ ]:
# ── 저장소 루트로 이동: 노트북 위치와 무관하게 data/·outputs/ 상대경로 유지 ──
import os
from pathlib import Path
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / 'README.md').exists() and (_c / '.gitignore').exists():
        os.chdir(_c); break


# 02. 도로망 구축 (M1) — 차량 그래프 뼈대

## 이 노트북이 하는 일
OpenStreetMap에서 **차량 도로망**을 받아 노드(교차로)·엣지(도로)의 그래프를 만들고, 병원·119안전센터·발생지를 도로망에 스냅한다. 이후 경사(03)·연결성(06)이 이 위에 쌓인다.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 OSM인가:** 위상(topology)이 이미 구축돼 있어 바로 그래프로 쓸 수 있다. 수치지형도 도로선은 교차로에서 안 끊겨 별도 위상작업이 필요.
- **왜 차량망만(우선):** 구급차 접근이 핵심. 계단(steps)은 drive 망에서 자동 제외된다. 보행망은 추후.
- **왜 graph_area를 넓게(볼록껍질+1.5km 버퍼) 잡나:** 그래프를 대상지로 자르면 **경계효과**로 도달성이 과대추정된다. "계산은 넓게, 해석은 좁게".
- **왜 EPSG:5186으로 투영하나:** 위경도(4326)는 각도라 거리 계산이 부정확. 부산권 평면좌표계(5186)로 바꿔야 미터 단위 거리가 정확.
- **왜 스냅하나:** 구급차는 직선이 아니라 도로를 따라 이동. 병원·발생지를 '가장 가까운 도로노드'에 붙여야 Dijkstra가 도로 경로로 계산.
- **왜 time을 아직 안 넣나:** 경사·도로폭 반영 실제 도달시간은 후속 과제. 여기선 거리(length) 기준 임시 경로만.

## 데이터 출처
- 병원 3곳: 공공데이터포털 15000563 API(01에서 확인) — 확정값 하드코딩
- 도로망·경계·안전센터: OpenStreetMap © OpenStreetMap contributors (ODbL)

In [ ]:
# 최초 1회 설치 (이미 있으면 건너뜀)
%pip install osmnx geopandas networkx folium shapely pyproj pandas pyarrow

In [ ]:
import os, warnings                       # os: 폴더 생성 / warnings: 지도라이브러리 경고 숨김
warnings.filterwarnings("ignore")         # geopandas/osmnx의 잦은 경고를 출력에서 제거(가독성)
import osmnx as ox                         # OSM에서 도로망·시설을 받아 networkx 그래프로 만드는 핵심 라이브러리
import geopandas as gpd                    # 좌표계 변환·공간연산이 되는 표(GeoDataFrame)
import networkx as nx                      # 그래프 자료구조·최단경로(Dijkstra)
import pandas as pd                        # 일반 표 처리
from shapely.geometry import Point, box    # Point: 점 좌표 / box: 사각형 폴리곤
from shapely.ops import unary_union        # 여러 도형을 하나로 합치기(경계+병원+센터)
import folium                              # 브라우저에서 보는 인터랙티브 지도

print("osmnx", ox.__version__)             # 버전 기록(재현성)
CRS_WGS = 4326                             # 위경도 좌표계(OSM 원본)
CRS_M   = 5186                             # 부산권 평면좌표계(거리계산용, 단위 m)
BUFFER_M = 1500                            # graph_area 버퍼 1.5km — 우회경로가 껍질 밖으로 나가는 최대폭 근사
os.makedirs("outputs", exist_ok=True)      # 산출물 폴더 준비(없으면 생성)

## 1. 입력 — 이송 후보 병원 3곳 (01·축3에서 확정)

In [ ]:
hospitals = [                                                                    # 01에서 등급 확인해 고른 센터급 3곳
    {"name":"동아대학교병원",   "grade":"권역응급의료센터", "lon":129.017604, "lat":35.120006, "hpid":"A1200003"},
    {"name":"부산대학교병원",   "grade":"지역응급의료센터", "lon":129.019222, "lat":35.101054, "hpid":"A1200002"},
    {"name":"인제대부산백병원", "grade":"지역응급의료센터", "lon":129.020572, "lat":35.146454, "hpid":"A1200001"},
]
hosp_gdf = gpd.GeoDataFrame(                                                      # dict 리스트를 지리 표로 변환
    hospitals,
    geometry=[Point(h["lon"], h["lat"]) for h in hospitals],                     # 경위도로 점 생성
    crs=CRS_WGS,                                                                 # 원본은 위경도(4326)
)
hosp_gdf[["name","grade","lon","lat"]]                                            # 확인용 표 출력

## 2. 초량·좌천 행정동 경계 (OSM 행정경계 릴레이션)

In [ ]:
DONGGU_BBOX = box(129.020, 35.100, 129.075, 35.155)   # 동구를 넉넉히 덮는 사각형(경계 조회 범위)
boundary_geom, boundary_src = None, None              # 결과를 담을 변수 초기화
try:
    adm = ox.features_from_polygon(DONGGU_BBOX, tags={"boundary":"administrative"})  # 이 범위 내 행정경계 전부 조회
    adm = adm[adm.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()        # 면(폴리곤)만 남김(점/선 제거)
    nm = adm["name"].astype(str) if "name" in adm.columns else pd.Series([""]*len(adm), index=adm.index)  # 이름 컬럼 안전 확보
    sel = adm[nm.str.contains("초량|좌천", na=False)]                                 # 이름에 '초량' 또는 '좌천' 포함만 필터
    if len(sel) > 0:                                                                  # 하나라도 찾으면
        boundary_geom = unary_union(sel.geometry)                                     # 여러 동 폴리곤을 하나로 합침
        boundary_src  = "OSM 행정경계 (%d개 폴리곤: %s)" % (len(sel), ", ".join(sorted(set(nm[sel.index]))))
except Exception as e:                                                                # 조회 실패 대비
    print("행정경계 조회 실패:", e)
if boundary_geom is None:                                                             # 못 찾았으면 fallback 사각형 사용
    boundary_geom = box(129.030, 35.110, 129.060, 35.145)
    boundary_src  = "fallback bbox"
boundary = gpd.GeoDataFrame(geometry=[boundary_geom], crs=CRS_WGS)                    # 경계를 지리 표로 감쌈
print("경계 출처:", boundary_src)                                                     # OSM 성공/ fallback 여부 기록

## 3. 119안전센터 (OSM amenity=fire_station)\n구급차 배치 여부는 미포함 → 추후 소방청 데이터로 보정.

In [ ]:
base = unary_union([boundary_geom] + list(hosp_gdf.geometry))                        # 경계+병원을 합친 도형
search_poly = (gpd.GeoSeries([base], crs=CRS_WGS).to_crs(CRS_M)                       # 5186으로 바꿔(미터)
               .convex_hull.buffer(2000).to_crs(CRS_WGS).iloc[0])                     # 볼록껍질+2km 버퍼 후 다시 위경도 → 소방서 검색영역
try:
    fs = ox.features_from_polygon(search_poly, tags={"amenity":"fire_station"})       # 검색영역 내 소방/안전센터 조회
    fs = fs[fs.geometry.notna()].copy()                                              # 좌표 없는 항목 제거
    fs["geometry"] = fs.geometry.centroid                                            # 건물이 면이면 중심점으로 변환
    name_col = fs["name"] if "name" in fs.columns else ["119안전센터"]*len(fs)        # 이름 없으면 기본명
    stations = gpd.GeoDataFrame({"name": list(name_col)}, geometry=list(fs.geometry), crs=CRS_WGS)  # 지리 표로 정리
except Exception as e:
    print("fire_station 조회 실패:", e)                                               # 실패 시 빈 표
    stations = gpd.GeoDataFrame({"name":[]}, geometry=[], crs=CRS_WGS)
print("안전센터/소방 시설 수:", len(stations))                                         # 확보 개수
stations.head()

## 4. graph_area — (경계 ∪ 병원 ∪ 안전센터) 볼록껍질 + 1.5km 버퍼\n클리핑 금지: 경계효과 방지를 위해 넓게.

In [ ]:
pts = list(hosp_gdf.geometry) + list(stations.geometry)                              # 병원+안전센터 점 모음
merged = unary_union([boundary_geom] + pts)                                          # 경계와 점들을 하나로 합침
graph_area_m   = (gpd.GeoSeries([merged], crs=CRS_WGS).to_crs(CRS_M)                 # 5186(미터)로 변환 후
                  .convex_hull.buffer(BUFFER_M).iloc[0])                             # 전체를 감싸는 볼록껍질 + 1.5km 버퍼
graph_area_wgs = gpd.GeoSeries([graph_area_m], crs=CRS_M).to_crs(CRS_WGS).iloc[0]    # OSM 추출용으로 위경도로 되돌림
print("graph_area 면적(㎢): %.2f" % (graph_area_m.area/1e6))                          # 넓이 확인(㎡→㎢)

## 5. OSM 차량(drive) 네트워크 추출\ncalc: network_type='drive' — 계단은 자동 제외.

In [ ]:
G  = ox.graph_from_polygon(graph_area_wgs, network_type="drive", simplify=True)      # 영역 내 차량 도로망을 그래프로 추출(단순화)
Gp = ox.project_graph(G, to_crs=f"EPSG:{CRS_M}")                                     # 거리계산 위해 5186으로 투영한 사본
print("노드:", Gp.number_of_nodes(), "| 엣지:", Gp.number_of_edges())                # 규모 확인

## 6. 노드·엣지 속성 채우기 (time 제외)
엣지: length(존재)·hw_type(도로유형)·width_est(폭 추정)·amb_passable(구급차 통과 예비). 경사·표고는 DEM 필요 → 03에서.

In [ ]:
WIDTH_DEFAULT = {                                                                    # OSM width 태그가 대부분 비어 있어, 도로유형별 기본 폭(m) 추정치
    "motorway":12,"trunk":10,"primary":10,"secondary":8,"tertiary":7,
    "unclassified":5,"residential":5,"living_street":4,"service":3.5,"road":5,
    "motorway_link":7,"trunk_link":7,"primary_link":7,"secondary_link":6,"tertiary_link":6,
}
def first(v):                                                                        # OSM 태그가 리스트로 올 때가 있어 첫 값만 취함
    return v[0] if isinstance(v, list) else v

for u, v, k, d in Gp.edges(keys=True, data=True):                                     # 모든 엣지를 순회
    hw = first(d.get("highway", "road"))                                             # 도로유형(없으면 'road')
    d["hw_type"] = hw                                                                 # 유형 저장
    w = d.get("width")                                                               # OSM에 실제 폭 태그가 있으면
    try:
        w = float(first(w)) if w is not None else None                               # 숫자로 변환
    except Exception:
        w = None                                                                     # 변환 실패면 없음 처리
    if w is None:                                                                     # 폭 태그 없으면
        w = WIDTH_DEFAULT.get(hw, 5.0)                                               # 유형별 기본값 사용
    d["width_est"]    = float(w)                                                      # 추정 폭 저장
    d["amb_passable"] = bool(w >= 3.0)                                               # 폭 3m 미만이면 구급차 진입 곤란 → 예비 판정
    d["grade"]  = ""                                                                  # 경사 자리(03에서 채움)
    d["elev_u"] = ""                                                                  # 시작노드 표고 자리
    d["elev_v"] = ""                                                                  # 끝노드 표고 자리
    # time 은 여기서 계산하지 않음 (경사·폭 반영 도달시간은 후속 과제)

for n, d in Gp.nodes(data=True):                                                     # 모든 노드를 순회
    d["node_type"] = "road"                                                           # 기본 유형 '일반 도로노드'
    d["snap_ref"]  = ""                                                               # 어떤 병원/센터가 스냅됐는지 자리
    d["elev"]      = ""                                                               # 표고 자리(03에서)

pd.Series([d["hw_type"] for *_ , d in Gp.edges(keys=True, data=True)]).value_counts()  # 도로유형 분포 확인

## 7. 병원·안전센터·발생지 스냅 (최근접 도로노드)

In [ ]:
hosp_m = hosp_gdf.to_crs(CRS_M)                                                       # 병원 좌표를 5186으로 변환(최근접 계산용)
hosp_nodes = ox.distance.nearest_nodes(Gp, X=hosp_m.geometry.x.values, Y=hosp_m.geometry.y.values)  # 각 병원의 최근접 도로노드
hosp_nodes = list(hosp_nodes)
for nid, name in zip(hosp_nodes, hosp_gdf["name"]):                                   # 스냅된 노드에 표시
    Gp.nodes[nid]["node_type"] = "hospital"                                           # 유형=병원
    Gp.nodes[nid]["snap_ref"]  = name                                                 # 어느 병원인지 기록

station_nodes = []
if len(stations) > 0:                                                                 # 안전센터가 있으면
    st_m = stations.to_crs(CRS_M)                                                     # 5186 변환
    station_nodes = list(ox.distance.nearest_nodes(Gp, X=st_m.geometry.x.values, Y=st_m.geometry.y.values))  # 최근접 노드
    for nid in station_nodes:
        if Gp.nodes[nid]["node_type"] == "road":                                      # 병원과 겹치지 않은 노드만
            Gp.nodes[nid]["node_type"] = "station"                                    # 유형=안전센터
            Gp.nodes[nid]["snap_ref"]  = "119안전센터"

ORIGIN_LONLAT = (129.0375, 35.1205)                                                   # 샘플 발생지(초량 산복도로 한 지점)
o_m = gpd.GeoSeries([Point(*ORIGIN_LONLAT)], crs=CRS_WGS).to_crs(CRS_M)               # 5186 변환
origin_node = int(ox.distance.nearest_nodes(Gp, X=o_m.geometry.x.values[0], Y=o_m.geometry.y.values[0]))  # 최근접 노드
Gp.nodes[origin_node]["node_type"] = "incident"                                       # 유형=발생지
Gp.nodes[origin_node]["snap_ref"]  = "샘플 발생지(초량 산복도로)"
print("병원 노드:", hosp_nodes)
print("안전센터 노드:", station_nodes)
print("발생지 노드:", origin_node)

## 8. 샘플 경로 (거리 기준 임시)
time이 아직 없으므로 length(도로거리)로 최단경로를 뽑아 지도 확인용으로만 사용.

In [ ]:
def nearest_route(src_candidates, dst, weight="length"):                              # 여러 출발후보 중 dst에 가장 가까운 경로 찾기
    best = None
    for s in src_candidates:                                                          # 후보마다
        try:
            L = nx.shortest_path_length(Gp, s, dst, weight=weight)                    # 최단거리 계산
            if best is None or L < best[0]:                                           # 가장 짧은 것 갱신
                best = (L, s)
        except nx.NetworkXNoPath:                                                     # 경로 없으면 건너뜀
            continue
    if best is None:
        return None, None
    return nx.shortest_path(Gp, best[1], dst, weight=weight), best[0]                 # 최적 출발점의 경로 반환

route_transport, Lt = nearest_route(hosp_nodes, origin_node)                          # 발생지에 가장 가까운 병원 찾기(역방향)
if route_transport is not None:
    dst_h = route_transport[0]                                                        # 그 병원 노드
    route_transport = nx.shortest_path(Gp, origin_node, dst_h, weight="length")       # 발생지 → 병원 방향 경로로 재계산
    Lt = nx.shortest_path_length(Gp, origin_node, dst_h, weight="length")
    print("이송 경로: 발생지 → %s | 도로거리 %.0f m" % (Gp.nodes[dst_h]["snap_ref"], Lt))

route_dispatch, Ld = (None, None)
if station_nodes:                                                                     # 안전센터가 있으면
    route_dispatch, Ld = nearest_route(station_nodes, origin_node)                    # 최근접 센터 → 발생지 출동 경로
    if route_dispatch is not None:
        print("출동 경로: %s → 발생지 | 도로거리 %.0f m" % (Gp.nodes[route_dispatch[0]]["snap_ref"], Ld))
else:
    print("안전센터 노드 없음 → 출동 경로 생략")

## 9. 시각화 (folium)

In [ ]:
ctr = [35.122, 129.045]                                                              # 지도 중심(대략 초량)
m = folium.Map(location=ctr, zoom_start=14, tiles="cartodbpositron")                  # 밝은 배경 지도 생성

folium.GeoJson(gpd.GeoSeries([graph_area_wgs], crs=CRS_WGS).__geo_interface__,        # graph_area 경계(회색)
               name="graph_area",
               style_function=lambda x: {"color":"#888","weight":1,"fillOpacity":0.04}).add_to(m)
folium.GeoJson(gpd.GeoSeries([boundary_geom], crs=CRS_WGS).__geo_interface__,         # 초량·좌천 경계(파랑)
               name="초량·좌천 경계",
               style_function=lambda x: {"color":"#2c7fb8","weight":2,"fill":False}).add_to(m)

edges_wgs = ox.graph_to_gdfs(G, nodes=False)[["geometry"]]                            # 도로망 엣지를 위경도로 뽑음
folium.GeoJson(edges_wgs.to_json(), name="차량 도로망",                                # 도로(연회색 선)
               style_function=lambda x: {"color":"#c0c0c0","weight":1}).add_to(m)

for _, h in hosp_gdf.iterrows():                                                      # 병원 마커(빨강 +)
    folium.Marker([h["lat"], h["lon"]], tooltip=f"{h['name']} ({h['grade']})",
                  icon=folium.Icon(color="red", icon="plus", prefix="fa")).add_to(m)
for _, s in stations.iterrows():                                                      # 안전센터 마커(파랑 불꽃)
    folium.Marker([s.geometry.y, s.geometry.x], tooltip=str(s["name"]),
                  icon=folium.Icon(color="blue", icon="fire", prefix="fa")).add_to(m)
folium.Marker([ORIGIN_LONLAT[1], ORIGIN_LONLAT[0]], tooltip="샘플 발생지",             # 발생지 마커(검정 별)
              icon=folium.Icon(color="black", icon="star", prefix="fa")).add_to(m)

def rcoords(route):                                                                   # 노드 경로를 [(위도,경도),...]로 변환(선 그리기용)
    return [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in route]
if route_transport:                                                                   # 이송 경로 선(빨강)
    folium.PolyLine(rcoords(route_transport), color="crimson", weight=4, opacity=0.9,
                    tooltip="이송: 발생지→병원 (거리 기준)").add_to(m)
if route_dispatch:                                                                    # 출동 경로 선(초록)
    folium.PolyLine(rcoords(route_dispatch), color="green", weight=4, opacity=0.9,
                    tooltip="출동: 센터→발생지 (거리 기준)").add_to(m)

folium.LayerControl().add_to(m)                                                       # 레이어 토글 버튼
m.save("outputs/graph_drive_map.html")                                               # 지도 파일 저장
print("지도 저장: outputs/graph_drive_map.html")
m                                                                                     # 노트북에 지도 표시

## 10. 저장 (GraphML + GeoPackage + Parquet)

In [ ]:
ox.save_graphml(Gp, "outputs/graph_drive_M1.graphml")                                 # 위상 보존 재로드용(다음 노트북이 읽음)
try:
    ox.save_graph_geopackage(Gp, "outputs/graph_drive_M1.gpkg")                       # QGIS·지도용(nodes/edges 레이어)
except Exception as e:
    print("gpkg 저장 경고:", e)

nodes_gdf, edges_gdf = ox.graph_to_gdfs(Gp)                                           # 그래프를 노드/엣지 표로 분리
def stringify(gdf):                                                                   # parquet은 리스트·혼합타입 저장을 싫어함 → 문자열화
    g = gdf.copy()
    for c in g.columns:
        if c == "geometry":                                                          # 도형 컬럼은 그대로 둠
            continue
        if g[c].apply(lambda v: isinstance(v, (list, tuple))).any():                  # 리스트/튜플이 섞인 컬럼은
            g[c] = g[c].apply(lambda v: ";".join(map(str, v)) if isinstance(v, (list, tuple)) else v)  # 문자열로 합침
        if g[c].dtype == object:                                                      # 남은 object 컬럼(osmid 등)도
            g[c] = g[c].astype(str)                                                   # 안전하게 문자열화
    return g
for g, nm in [(nodes_gdf, "nodes"), (edges_gdf, "edges")]:                            # 노드·엣지 각각 parquet 저장
    try:
        stringify(g).to_parquet(f"outputs/{nm}_drive_M1.parquet")                     # 타입 보존 분석용
    except Exception as e:
        print(nm, "parquet 경고:", e)

print("저장 완료:", os.listdir("outputs"))                                             # 저장된 파일 목록 확인